In [1]:
import os

proj_data_dir = r"C:\Users\Gab\anaconda3\envs\gee_env\Library\share\proj"
os.environ.setdefault("PROJ_LIB", proj_data_dir)
os.environ.setdefault("PROJ_DATA", proj_data_dir)

import numpy as np
import xarray as xr
import ee
import geemap
import xee
from xee import helpers
import geopandas as gpd
import fiona
import matplotlib.pyplot as plt


c:\Users\Gab\anaconda3\envs\gee_env\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [2]:
ee.Authenticate()
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

c:\Users\Gab\anaconda3\envs\gee_env\Lib\site-packages\ee\deprecation.py:140: UserWarning: Unable to initialize deprecated assets: [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4057)
  warnings.warn(f'Unable to initialize deprecated assets: {e}')


In [3]:
# Caminho com os dados
gpkg_path = '../data/BHSF.gpkg'

# Verificar qual a layer do gpkg
fiona.listlayers(gpkg_path)


['bacia_SF', 'bacias_meso_SF', 'bacias_micro_SF', 'layer_styles']

In [5]:
# Carregar camada
gdf = gpd.read_file(gpkg_path, layer="bacias_meso_SF", engine="fiona")
out_json = "../outputs/bacias_meso_SF.geojson"
gdf.to_file(out_json, driver="GeoJSON")
geom_ee = geemap.geojson_to_ee(out_json)
area = geom_ee.geometry()


c:\Users\Gab\anaconda3\envs\gee_env\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect PROJ data files. Set PROJ_LIB environment variable to the correct path.
  _init_proj_data()


In [38]:
map = geemap.Map()
map.addLayer(area, {'color': 'blue', 'fillColor': '00000000'}, 'Bacias Meso SF')

# 3. Centralizar o mapa automaticamente na geometria da bacia
map.centerObject(area, zoom=6)

# 4. Exibir o mapa no notebook
map

Map(center=[-13.405889806359472, -43.19761216544879], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
map = map.draw

In [ ]:
import os
import ssl
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pyproj
import dask
import dask.base as dask_base

from xee import helpers

# Prevent Dask from probing distributed, which triggers an SSL failure in this environment.
dask_base._DISTRIBUTED_AVAILABLE = False
dask_base._distributed_available = lambda: False

# Avoid Windows certificate-store loading during optional distributed imports.
_original_create_default_context = ssl.create_default_context

def _safe_create_default_context(*args, **kwargs):
    context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    context.check_hostname = False
    context.verify_mode = ssl.CERT_NONE
    return context

ssl.create_default_context = _safe_create_default_context

proj_data_dir = pyproj.datadir.get_data_dir()
os.environ["PROJ_LIB"] = proj_data_dir
os.environ["PROJ_DATA"] = proj_data_dir
pyproj.datadir.set_data_dir(proj_data_dir)

# 1. Parâmetros de processamento e resolução
START_DATE = "2023-01-01"
END_DATE = "2024-12-31"
COLLECTION_ID = "MODIS/061/MOD09A1"
BANDS = ["sur_refl_b01", "sur_refl_b02", "sur_refl_b03", "sur_refl_b04", "sur_refl_b05", "sur_refl_b07"]
MODIS_SCALE = 0.0001
GRID_CRS = "+proj=cea +lon_0=0 +lat_ts=0 +datum=WGS84 +units=m +no_defs"
AOI_CRS = "+proj=longlat +datum=WGS84 +no_defs"
GRID_SCALE = (1000, -1000)
DASK_CHUNKS = {"time": 1, "y": 512, "x": 512}
IO_CHUNKS = {"time": 1, "y": 512, "x": 512}

# 2. Derivar a grade a partir da geometria já carregada no notebook
# Evitamos reabrir o GeoJSON para não disparar outra conversão CRS no GeoPandas.
aoi = gdf.geometry.union_all()
grid_params = helpers.fit_geometry(
    geometry=aoi,
    geometry_crs=AOI_CRS,
    grid_crs=GRID_CRS,
    grid_scale=GRID_SCALE,
)
print("Grid params:", grid_params)

# 3. Abrir a coleção do GEE via Xee com avaliação preguiçosa
ic = (
    ee.ImageCollection(COLLECTION_ID)
    .filterBounds(area)
    .filterDate(START_DATE, END_DATE)
    .select(BANDS)
)

ds = xr.open_dataset(
    ic,
    engine="ee",
    chunks=DASK_CHUNKS,
    io_chunks=IO_CHUNKS,
    mask_and_scale=False,
    n_images=128,
    **grid_params,
)
print(ds)

# 4. Aplicar os fatores de escala oficiais do sensor e calcular os índices
red = ds["sur_refl_b01"] * MODIS_SCALE
nir = ds["sur_refl_b02"] * MODIS_SCALE
b3 = ds["sur_refl_b03"] * MODIS_SCALE
b4 = ds["sur_refl_b04"] * MODIS_SCALE
b5 = ds["sur_refl_b05"] * MODIS_SCALE
b7 = ds["sur_refl_b07"] * MODIS_SCALE

radicand = ((2 * nir + 1) ** 2 - 8 * (nir - red)).clip(min=0)
msavi = ((2 * nir + 1 - np.sqrt(radicand)) / 2).rename("MSAVI")
albedo = (
    0.160 * red
    + 0.291 * nir
    + 0.243 * b3
    + 0.116 * b4
    + 0.112 * b5
    + 0.081 * b7
    - 0.0015
).rename("Albedo")

indices = xr.Dataset({"MSAVI": msavi, "Albedo": albedo})
print(indices)

# 5. Reduzir o tempo com mediana anual e depois obter uma superfície final 2D
with dask.config.set({"array.rechunk.method": "tasks"}):
    annual = indices.resample(time="1YE").median()
    median_maps = annual.median(dim="time")
print(median_maps)

# 6. Validação explícita de nulos e infinitos antes da plotagem
for name in ["MSAVI", "Albedo"]:
    da = median_maps[name]
    nulls = int(da.isnull().sum().compute().item())
    infs = int(np.isinf(da).sum().compute().item())
    print(f"{name} -> nulls: {nulls}, infs: {infs}")

# 7. Plotagem limpa em duas subplots
msavi_plot = median_maps["MSAVI"].compute()
albedo_plot = median_maps["Albedo"].compute()

fig, axes = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)
msavi_plot.plot(ax=axes[0], cmap="YlGn", robust=True, add_colorbar=True)
axes[0].set_title("MSAVI mediano")
axes[0].set_xlabel("X")
axes[0].set_ylabel("Y")

albedo_plot.plot(ax=axes[1], cmap="copper", robust=True, add_colorbar=True)
axes[1].set_title("Albedo mediano")
axes[1].set_xlabel("X")
axes[1].set_ylabel("Y")

plt.show()


c:\Users\Gab\anaconda3\envs\gee_env\Lib\site-packages\pyproj\datadir.py:38: UserWarning: pyproj unable to set PROJ database path.
  _set_context_data_dir()


Grid params: {'crs': '+proj=cea +lon_0=0 +lat_ts=0 +datum=WGS84 +units=m +no_defs', 'crs_transform': (1000.0, 0.0, -5304000.0, 0.0, -1000.0, -801000.0), 'shape_2d': (1263, 1464)}
<xarray.Dataset> Size: 4GB
Dimensions:       (time: 92, y: 1464, x: 1263)
Coordinates:
  * time          (time) datetime64[ms] 736B 2023-01-01 ... 2024-12-26
  * y             (y) float64 12kB -8.015e+05 -8.025e+05 ... -2.264e+06
  * x             (x) float64 10kB -5.304e+06 -5.302e+06 ... -4.042e+06
Data variables:
    sur_refl_b01  (time, y, x) float32 680MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    sur_refl_b02  (time, y, x) float32 680MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    sur_refl_b03  (time, y, x) float32 680MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    sur_refl_b04  (time, y, x) float32 680MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    sur_refl_b05  (time, y, x) float32 680MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    sur_refl_

SSLError: [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4057)